In [1]:
# Runtime evidence: this cell checks the environment actually used by the kernel.
import sys as _shape_sys, json as _shape_json, importlib.util as _shape_util
print("SHAPE_RUNTIME_PROVENANCE=" + _shape_json.dumps({
    "python_executable": _shape_sys.executable,
    "python_version": _shape_sys.version.split()[0],
    "shape_lab_origin": _shape_util.find_spec("shape_lab").origin
}))
del _shape_sys, _shape_json, _shape_util


SHAPE_RUNTIME_PROVENANCE={"python_executable": "/opt/pyvenv/bin/python", "python_version": "3.13.5", "shape_lab_origin": "/tmp/shape-course-q90rdcpz/course/src/shape_lab/__init__.py"}


# Stage 13 consolidation — A chart changes coordinates, not the underlying point

Read `sessions/13_session.md` first. This is an executed reference, not an assessment of you.

**Evidence:** Real stereo-reference directions mapped onto a constructed unit sphere; not a spherical scene claim.

In [2]:
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic('matplotlib','inline')
from pathlib import Path
import sys, json
import numpy as np

ROOT = Path.cwd().resolve()
while not (ROOT/'data/raw/iris.csv').is_file():
    if ROOT == ROOT.parent:
        raise FileNotFoundError('Start Jupyter in the extracted listening_to_shape_lab folder.')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT/'src'))


## Step 1

Convert real reference scene points into unit directions, explicitly discarding range.

Before running: write your expected result and name one way this check could fail.

In [3]:
from shape_lab.stereo import reference_sample, point_cloud
from shape_lab.geometry import sphere_chart, sphere_unchart
_, _, disparity, calibration = reference_sample()
points, rc = point_cloud(disparity, calibration, stride=50)
p = points/np.linalg.norm(points, axis=1, keepdims=True)
p = p[np.abs(p[:,2]) < .99][:40]
assert len(p)>3
print('unit directions:', len(p))

unit directions: 40


## Step 2

Use both charts and recover the same sphere point.

Before running: write your expected result and name one way this check could fail.

In [4]:
north = sphere_chart(p, 'north'); south = sphere_chart(p, 'south')
assert np.allclose(sphere_unchart(north, 'north'), p)
assert np.allclose(sphere_unchart(south, 'south'), p)
q = np.sum(north*north, axis=1)
assert np.allclose(north/q[:,None], south)
print('maximum round-trip error:', np.max(np.abs(sphere_unchart(north, 'north')-p)))

maximum round-trip error: 8.604228440844963e-16


## Step 3

Project arbitrary vectors onto the tangent planes and verify their radial component vanishes.

Before running: write your expected result and name one way this check could fail.

In [5]:
a = np.random.default_rng(1304).normal(size=p.shape)
v = a - np.sum(a*p, axis=1)[:,None]*p
residual = float(np.max(np.abs(np.sum(p*v, axis=1))))
assert residual < 1e-12
result = {'direction_count': len(p), 'chart_round_trip': 'passed',
          'tangent_orthogonality_max': residual, 'range_retained': False,
          'physical_scene_is_sphere_claimed': False}

## Keep execution evidence separate from learning evidence

The following saves this reference calculation. Write your own explanation in `my_work/`; no learner score is assigned.

In [6]:
result['stage'] = 13
result['execution_role'] = 'reference_consolidation'
result['data_role'] = 'Real stereo-reference directions mapped onto a constructed unit sphere; not a spherical scene claim.'
output = ROOT/'reports/consolidation_v4/13.json'
output.parent.mkdir(parents=True, exist_ok=True)
output.write_text(json.dumps(result, indent=2, allow_nan=False) + '\n')
print('Saved reference result:', output.relative_to(ROOT))


Saved reference result: reports/consolidation_v4/13.json
